In [25]:
import os
import gzip
import pickle
import glob
import torch
from metient.util.globals import *
from metient import metient as met
from metient.util.data_extraction_util import adjacency_matrix_from_parents


In [26]:
REPO_DIR = '/lila/data/morrisq/divyak/projects/metient/data'
OUTPUT_DIR = os.path.join(REPO_DIR, '../notebooks', 'output_plots')


NUM_RUNS = 5
DATE = "04162025"
DATASET_NAMES = [ "HGSOC", "Melanoma", "HR-NB", "NSCLC"]
CALIBRATE_DIRS = [os.path.join(REPO_DIR,f"mcpherson_ovarian_2016/metient_outputs/hgsoc_bs8192_runs100_solvepolys_{DATE}"),
                  os.path.join(REPO_DIR,f"sanborn_melanoma_2015/metient_outputs/melanoma_bs8192_runs100_solvepolys_{DATE}"),
                  os.path.join(REPO_DIR,f"gundem_neuroblastoma_2023/metient_outputs/hrnb_bs8192_runs100_solvepolys_{DATE}"),
                  os.path.join(REPO_DIR,f"tracerx_nsclc/metient_outputs/tracerx_trees_bs8192_runs100_{DATE}",)]
SOLVE_POLYS = True

NUM_RUNS = 3
DATE = "05142025"
DATASET_NAMES = [ "LT"]
CALIBRATE_DIRS = [os.path.join(REPO_DIR,f"quinn_lt_2021/metient_outputs", f"2048bs_200runs_{DATE}"),]
SOLVE_POLYS = False


### Load the parsimony metrics from all runs

In [27]:

class TreeAndLabel:
    def __init__(self, parents, labels, loss, mig_graph, node_info):
        """
        Initialize a tree and labels object
        
        Args:
            parents: Array where index i contains parent of node i
            labels: k x n tensor of labels, where k is number of mutations and n is number of cells
        """
        if not isinstance(parents, torch.Tensor):
            parents = torch.tensor(parents)
        self.parents = parents

        if not isinstance(labels, torch.Tensor):
            labels = torch.tensor(labels)
        self.labels = labels

        self.loss = loss
        self.mig_graph = mig_graph
        # A = adjacency_matrix_from_parents(self.parents)
        # self.genetic_clonality = met.genetic_clonality(self.labels, A, node_info)

        
    def __eq__(self, other):
        if not isinstance(other, TreeAndLabel):
            return False
        if self.parents.size() != other.parents.size() or self.labels.size() != other.labels.size():
            return False
            
        if not SOLVE_POLYS:
            # If not solving polytomies, just check if labels match exactly since tree structure is fixed
            return torch.equal(self.labels, other.labels)
            
        # Otherwise need to check for isomorphism
        # Convert both trees to networkx graphs
        import networkx as nx
        
        # Create graphs from parent arrays
        g1 = nx.DiGraph()
        g2 = nx.DiGraph()
        
        # Add nodes with label attributes
        for i in range(len(self.parents)):
            label_vec = tuple(self.labels[:, i].tolist())
            g1.add_node(i, label=label_vec)
        for i in range(len(other.parents)):
            label_vec = tuple(other.labels[:, i].tolist())
            g2.add_node(i, label=label_vec)
            
        # Add edges based on parent arrays
        for i, p in enumerate(self.parents):
            if p >= 0:  # Skip root which has parent -1
                g1.add_edge(int(p), i)
        for i, p in enumerate(other.parents):
            if p >= 0:
                g2.add_edge(int(p), i)
                
        # Use networkx's isomorphism checker that considers node attributes
        matcher = nx.isomorphism.DiGraphMatcher(
            g1, g2,
            node_match=lambda n1, n2: n1['label'] == n2['label']
        )
        
        # Check if isomorphic and get the node mapping
        if matcher.is_isomorphic():
            # Get the mapping between nodes
            node_mapping = matcher.mapping
            
            # Verify that mapped nodes have same labels
            for node1, node2 in node_mapping.items():
                if not torch.equal(self.labels[:, node1], other.labels[:, node2]):
                    return False
            return True
            
        return False
        
    def __hash__(self):
        # Convert parents tensor to tuple for hashing
        parents_tuple = tuple(self.parents.tolist())
        # Convert labels tensor to tuple for hashing 
        labels_tuple = tuple(self.labels.flatten().tolist())
        return hash((parents_tuple, labels_tuple))

def extract_all_parsimony_metrics(loss_dicts):
    all_pars_metrics = []
    for loss_dict in loss_dicts:
        all_pars_metrics.append((int(loss_dict[MIG_KEY]), int(loss_dict[COMIG_KEY]), int(loss_dict[SEEDING_KEY])))
    return tuple(all_pars_metrics)

def get_weighted_classifications(pkl):
    phyl = met.weighted_phyleticity(pkl)
    site_clonal = met.weighted_site_clonality(pkl)
    gen_clonal = met.weighted_genetic_clonality(pkl)
    pattern = met.weighted_seeding_pattern(pkl)
    return (phyl, site_clonal, gen_clonal, pattern)

pars_metrics_dict = {}
labeling_dict = {}
classifications_dict = {}

for dataset_name, calibrate_dir in zip(DATASET_NAMES, CALIBRATE_DIRS):

    for run in range(1,NUM_RUNS+1):
        if dataset_name == "LT":
            run_dir = os.path.join(calibrate_dir + f"_r{run}")
        else:
            run_dir = os.path.join(calibrate_dir + f"_r{run}", "calibrate")
        print(run_dir)
        matching_files = glob.glob(f'{run_dir}/*pkl.gz')
        print("Loading", len(matching_files), "files for", dataset_name)
        for fn in matching_files:
            with gzip.open(fn, 'rb') as f:
                pkl = pickle.load(f)                
            run_name = fn.split("/")[-1].replace(".pkl.gz", "")
          
            loss_dicts = pkl[OUT_LOSS_DICT_KEY]
            As = pkl[OUT_ADJ_KEY] # parents vectors
            adj_matrices = [met.adjacency_matrix_from_parents(A) for A in As]
            Vs = pkl[OUT_LABElING_KEY]
            mig_graphs = [met.migration_graph(V, A) for V,A in zip(Vs,adj_matrices)]
            losses = [loss_dict[FULL_LOSS_KEY] for loss_dict in loss_dicts]
            node_infos = pkl[OUT_IDX_LABEL_KEY]
            all_pars_metrics = extract_all_parsimony_metrics(loss_dicts)
            if run_name not in pars_metrics_dict:
                pars_metrics_dict[run_name] = []
                labeling_dict[run_name] = []
                classifications_dict[run_name] = []
            pars_metrics_dict[run_name].append(all_pars_metrics)
            classifications_dict[run_name].append(get_weighted_classifications(pkl))
            labeling_dict[run_name].append(tuple([TreeAndLabel(A,V,loss,G,node_info) for A,V,loss,G,node_info in zip(As,Vs,losses,mig_graphs, node_infos)]))
           

/lila/data/morrisq/divyak/projects/metient/data/quinn_lt_2021/metient_outputs/2048bs_200runs_05142025_r1
Loading 100 files for LT


/lila/data/morrisq/divyak/projects/metient/metient/util/data_extraction_util.py:35: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  parents = torch.tensor(parents)[mask]


/lila/data/morrisq/divyak/projects/metient/data/quinn_lt_2021/metient_outputs/2048bs_200runs_05142025_r2
Loading 100 files for LT
/lila/data/morrisq/divyak/projects/metient/data/quinn_lt_2021/metient_outputs/2048bs_200runs_05142025_r3
Loading 100 files for LT


### Calculate the consistency percentage of getting the exact same Pareto front metrics, in the same order

In [28]:
run_name_to_consistency_percentage = {}
for run_name, metrics in pars_metrics_dict.items():
    mode_count = max([metrics.count(x) for x in metrics])

    mode_percentage = (mode_count / NUM_RUNS) * 100
    run_name_to_consistency_percentage[run_name] = mode_percentage

    if mode_percentage < 100:
        print(run_name, mode_percentage, metrics)

avg_consistency_percentage = sum(run_name_to_consistency_percentage.values()) / len(run_name_to_consistency_percentage)

print("Average consistency percentage:")
print(avg_consistency_percentage)



1_LL 33.33333333333333 [((4306, 324, 6), (6431, 255, 6), (6446, 232, 6), (6423, 282, 6), (6410, 310, 6), (6461, 231, 6), (6484, 199, 6), (6476, 216, 6), (6508, 175, 6), (6507, 186, 6), (6545, 164, 6), (6545, 164, 6), (6569, 163, 6), (6572, 161, 6), (6578, 153, 6), (6611, 136, 6), (6644, 120, 6), (6708, 113, 6), (6807, 100, 6), (6872, 95, 6), (6871, 99, 6), (6905, 93, 6), (6926, 92, 6), (6976, 87, 6), (6991, 81, 6), (7011, 77, 6), (7051, 74, 6), (7056, 73, 6), (7068, 72, 6), (7077, 67, 6), (7112, 66, 6), (7112, 66, 6), (7123, 64, 6), (7129, 58, 6), (7139, 56, 6), (7161, 55, 6), (7176, 48, 6), (7244, 45, 6), (7253, 44, 6), (7266, 37, 6), (7288, 48, 5), (7279, 114, 5), (7342, 45, 5), (7371, 36, 6), (7374, 32, 6), (7374, 41, 5), (7388, 38, 5), (7398, 31, 5), (7399, 30, 6), (7428, 28, 5), (7438, 26, 6), (7450, 23, 6), (7443, 47, 4), (7450, 39, 4), (7475, 25, 5), (7483, 26, 4), (7514, 21, 5), (7554, 18, 4), (7575, 17, 6), (7608, 13, 5), (7607, 16, 5), (7621, 11, 4), (7605, 47, 3), (7640, 14,

### Calculate the consistency percentage of getting the exact same Pareto front metrics but in a different order

In [29]:
run_name_to_consistency_percentage = {}
for run_name, metrics in pars_metrics_dict.items():
    sorted_metrics = [sorted(x) for x in metrics]
    mode_count = max([sorted_metrics.count(x) for x in sorted_metrics])
    mode_percentage = (mode_count / NUM_RUNS) * 100
    run_name_to_consistency_percentage[run_name] = mode_percentage

    if mode_percentage < 100:
        print(run_name, mode_percentage, metrics)

avg_consistency_percentage = sum(run_name_to_consistency_percentage.values()) / len(run_name_to_consistency_percentage)

print("Average consistency percentage:")
print(avg_consistency_percentage)

1_LL 33.33333333333333 [((4306, 324, 6), (6431, 255, 6), (6446, 232, 6), (6423, 282, 6), (6410, 310, 6), (6461, 231, 6), (6484, 199, 6), (6476, 216, 6), (6508, 175, 6), (6507, 186, 6), (6545, 164, 6), (6545, 164, 6), (6569, 163, 6), (6572, 161, 6), (6578, 153, 6), (6611, 136, 6), (6644, 120, 6), (6708, 113, 6), (6807, 100, 6), (6872, 95, 6), (6871, 99, 6), (6905, 93, 6), (6926, 92, 6), (6976, 87, 6), (6991, 81, 6), (7011, 77, 6), (7051, 74, 6), (7056, 73, 6), (7068, 72, 6), (7077, 67, 6), (7112, 66, 6), (7112, 66, 6), (7123, 64, 6), (7129, 58, 6), (7139, 56, 6), (7161, 55, 6), (7176, 48, 6), (7244, 45, 6), (7253, 44, 6), (7266, 37, 6), (7288, 48, 5), (7279, 114, 5), (7342, 45, 5), (7371, 36, 6), (7374, 32, 6), (7374, 41, 5), (7388, 38, 5), (7398, 31, 5), (7399, 30, 6), (7428, 28, 5), (7438, 26, 6), (7450, 23, 6), (7443, 47, 4), (7450, 39, 4), (7475, 25, 5), (7483, 26, 4), (7514, 21, 5), (7554, 18, 4), (7575, 17, 6), (7608, 13, 5), (7607, 16, 5), (7621, 11, 4), (7605, 47, 3), (7640, 14,

### Calculate the consistency percentage of getting the same Pareto optimal metrics

In [30]:
unique_pars_metrics_dict = {}
for run_name, metrics in pars_metrics_dict.items():
    unique_metrics = []
    for metric in metrics:
        unique_metrics.append(tuple(set(metric)))
    unique_pars_metrics_dict[run_name] = tuple(unique_metrics)

run_name_to_consistency_percentage = {}
for run_name, metrics in unique_pars_metrics_dict.items():
    mode_count = max([metrics.count(x) for x in metrics])
    mode_percentage = (mode_count / NUM_RUNS) * 100
    run_name_to_consistency_percentage[run_name] = mode_percentage

    if mode_percentage < 100:
        print(run_name, mode_percentage, metrics)

avg_consistency_percentage = sum(run_name_to_consistency_percentage.values()) / len(run_name_to_consistency_percentage)

print("Average consistency percentage:")
print(avg_consistency_percentage)

1_LL 33.33333333333333 (((6507, 186, 6), (7051, 74, 6), (6708, 113, 6), (6905, 93, 6), (7253, 44, 6), (7288, 48, 5), (6446, 232, 6), (7077, 67, 6), (7371, 36, 6), (7438, 26, 6), (7554, 18, 4), (7443, 47, 4), (7677, 10, 4), (7388, 38, 5), (4306, 324, 6), (6569, 163, 6), (7621, 11, 4), (7709, 9, 3), (6572, 161, 6), (6484, 199, 6), (7475, 25, 5), (7605, 47, 3), (7483, 26, 4), (6508, 175, 6), (7244, 45, 6), (7698, 14, 2), (7266, 37, 6), (7722, 5, 1), (6871, 99, 6), (7575, 17, 6), (6410, 310, 6), (7279, 114, 5), (7068, 72, 6), (6991, 81, 6), (7161, 55, 6), (6578, 153, 6), (7123, 64, 6), (7450, 23, 6), (6431, 255, 6), (6461, 231, 6), (7056, 73, 6), (7112, 66, 6), (7428, 28, 5), (6423, 282, 6), (6807, 100, 6), (7342, 45, 5), (7011, 77, 6), (7398, 31, 5), (7129, 58, 6), (7677, 13, 3), (6872, 95, 6), (7176, 48, 6), (6644, 120, 6), (7374, 32, 6), (7399, 30, 6), (7608, 13, 5), (7607, 16, 5), (6926, 92, 6), (7712, 10, 2), (6476, 216, 6), (7695, 12, 3), (7514, 21, 5), (7640, 14, 3), (7374, 41, 5), 

### Calculate the consistency percentage of getting the same exact TOP tree and labeling

In [31]:
run_name_to_consistency_percentage = {}

for run_name in labeling_dict:
    labelings = labeling_dict[run_name]

    first_labelings = [x[0] for x in labelings]
    mode_count = first_labelings.count(first_labelings[0])
    mode_percentage = (mode_count / NUM_RUNS) * 100

    # In some cases, the first and second labelings have the exact same loss value, so which
    # labeling is chosen as the first labeling is arbitrary.
    if mode_percentage < 100:
        mode_count = 0
        for labeling_tuple in labelings:
            equivalent_labelings = []
            first_loss = labeling_tuple[0].loss
            for labeling in labeling_tuple:
                if abs(labeling.loss - first_loss) < 1e-3:
                    equivalent_labelings.append(labeling)
            for equivalent_labeling in equivalent_labelings:
                if equivalent_labeling == first_labelings[0]:
                    mode_count += 1
        mode_percentage = (mode_count / NUM_RUNS) * 100

    run_name_to_consistency_percentage[run_name] = mode_percentage

    if mode_percentage < 100:
        print(run_name, mode_percentage)
avg_consistency_percentage = sum(run_name_to_consistency_percentage.values()) / len(run_name_to_consistency_percentage)

print("Average consistency percentage:")
print(avg_consistency_percentage)

16_LL 33.33333333333333
23_LL 66.66666666666666
35_LL 66.66666666666666
77_LL 33.33333333333333
36_LL 33.33333333333333
54_LL 66.66666666666666
26_LL 66.66666666666666
20_LL 33.33333333333333
32_LL 66.66666666666666
34_LL 33.33333333333333
10_LL 66.66666666666666
25_LL 66.66666666666666
Average consistency percentage:
94.33333333333331


In [32]:
len(labeling_dict.keys())

100

### Calculate the consistency percentage of getting the same exact tree and labelings

In [33]:
run_name_to_consistency_percentage = {}

for run_name in labeling_dict:
    labelings = labeling_dict[run_name]

    mode_count = max([labelings.count(x) for x in labelings])
    mode_percentage = (mode_count / NUM_RUNS) * 100
    run_name_to_consistency_percentage[run_name] = mode_percentage

  
    # In some cases,  labelings have the exact same loss value, so which
    # labeling is chosen as the first labeling is arbitrary.
    if mode_percentage < 100:
        mode_count = 0
        print("\n", run_name)
        new_labelings = []
        for labeling_tuple in labelings:
            losses = [l.loss for l in labeling_tuple]
            # Sort labeling tuple by loss first, then by parents, then by labels
            sorted_labeling = tuple(sorted(labeling_tuple, 
                key=lambda x: (round(x.loss, 1),
                             x.parents.detach().cpu().numpy().tobytes(),
                             x.labels.detach().cpu().numpy().tobytes())))
            new_labelings.append(sorted_labeling)
        mode_count = max([new_labelings.count(x) for x in new_labelings])
        mode_percentage = (mode_count / NUM_RUNS) * 100
        run_name_to_consistency_percentage[run_name] = mode_percentage
        print("mode_percentage", mode_percentage)
        
avg_consistency_percentage = sum(run_name_to_consistency_percentage.values()) / len(run_name_to_consistency_percentage)

print("Average consistency percentage:")
print(avg_consistency_percentage)


 1_LL
mode_percentage 33.33333333333333

 2_LL
mode_percentage 33.33333333333333

 84_LL
mode_percentage 66.66666666666666

 13_LL
mode_percentage 33.33333333333333

 43_LL
mode_percentage 33.33333333333333

 56_LL
mode_percentage 33.33333333333333

 16_LL
mode_percentage 33.33333333333333

 23_LL
mode_percentage 33.33333333333333

 11_LL
mode_percentage 33.33333333333333

 90_LL
mode_percentage 100.0

 35_LL
mode_percentage 33.33333333333333

 77_LL
mode_percentage 33.33333333333333

 45_LL
mode_percentage 66.66666666666666

 21_LL
mode_percentage 33.33333333333333

 24_LL
mode_percentage 33.33333333333333

 33_LL
mode_percentage 33.33333333333333

 31_LL
mode_percentage 33.33333333333333

 51_LL
mode_percentage 33.33333333333333

 37_LL
mode_percentage 33.33333333333333

 66_LL
mode_percentage 33.33333333333333

 6_LL
mode_percentage 33.33333333333333

 9_LL
mode_percentage 33.33333333333333

 64_LL
mode_percentage 33.33333333333333

 36_LL
mode_percentage 33.33333333333333

 83_LL


### Calculate the consistency percentage of normed migration graphs

In [34]:
import sys
sys.path.append(os.path.join(os.getcwd(), "notebooks/lineage_tracing"))
import lineage_tracing_utilities as lt
from metient.util import plotting_util as putil

run_name_to_consistency_percentage = {}


def normalized_migration_graph(weights, mig_graphs):
    # mig_graphs is a list of 2D tensors (migration graphs)
    # and weights is a list of scalars (weights for each migration graph)
    # Initialize an empty tensor for the weighted sum
    weighted_summed_tissue_trans = torch.zeros_like(mig_graphs[0])  # Same shape as the first matrix
    
    # Perform weighted summation
    for w, m in zip(weights, mig_graphs):
        weighted_summed_tissue_trans += w * m
    
    weighted_summed_tissue_trans.fill_diagonal_(0)
    row_sums = weighted_summed_tissue_trans.sum(dim=1, keepdim=True)
    # Replace zero sums with ones to avoid division by zero
    row_sums = row_sums.where(row_sums != 0, torch.tensor(1.0))
    # Divide each element by its corresponding row sum
    normed_tissue_trans = weighted_summed_tissue_trans / row_sums
    return normed_tissue_trans



def proportion_within_threshold(tensor_list, threshold=0.05):
    ref = tensor_list[0]
    total_elements = ref.shape[0] * ref.shape[1] * (len(tensor_list) - 1)
    within_thresh = 0

    for t in tensor_list[1:]:
        within_thresh += torch.sum(torch.abs(t - ref) <= threshold).item()
    return within_thresh / total_elements if total_elements > 0 else 1.0

for run_name in labeling_dict:
    labelings = labeling_dict[run_name]
    normed_mig_graphs = []
    
    for labeling_tuple in labelings:
        # normed_mig_graph = normalized_migration_graph(labeling_tuple)
        normed_mig_graph = normalized_migration_graph(
            putil.losses_to_probabilities([l.loss for l in labeling_tuple], temperature=1), 
            [l.mig_graph for l in labeling_tuple]
        )
        #print(weighted_scaled_migration_graph)
        normed_mig_graphs.append(normed_mig_graph)
    
    # mode_count = count_tensors_within_threshold(normed_mig_graphs)
    mode_percentage = (mode_count / NUM_RUNS) * 100

    mode_percentage = proportion_within_threshold(normed_mig_graphs) * 100
    run_name_to_consistency_percentage[run_name] = mode_percentage

    if mode_percentage < 100:
        print(run_name, mode_percentage)
        print(proportion_within_threshold(normed_mig_graphs))
        for tensor in normed_mig_graphs:
            print(tensor)
   
avg_consistency_percentage = sum(run_name_to_consistency_percentage.values()) / len(run_name_to_consistency_percentage)

print("Average consistency percentage:")
print(avg_consistency_percentage)
    

13_LL 83.33333333333334
0.8333333333333334
tensor([[0.00, 0.22, 0.33, 0.08, 0.18, 0.19],
        [0.48, 0.00, 0.14, 0.17, 0.09, 0.12],
        [0.56, 0.12, 0.00, 0.08, 0.13, 0.12],
        [0.39, 0.12, 0.26, 0.00, 0.13, 0.10],
        [0.44, 0.28, 0.15, 0.07, 0.00, 0.06],
        [0.32, 0.10, 0.41, 0.04, 0.12, 0.00]])
tensor([[0.00, 0.24, 0.35, 0.08, 0.17, 0.17],
        [0.47, 0.00, 0.17, 0.18, 0.07, 0.10],
        [0.57, 0.11, 0.00, 0.06, 0.13, 0.13],
        [0.39, 0.12, 0.26, 0.00, 0.13, 0.10],
        [0.48, 0.28, 0.14, 0.05, 0.00, 0.05],
        [0.32, 0.08, 0.41, 0.05, 0.14, 0.00]])
tensor([[0.00, 0.24, 0.34, 0.07, 0.16, 0.19],
        [0.28, 0.00, 0.06, 0.40, 0.13, 0.13],
        [0.68, 0.09, 0.00, 0.02, 0.10, 0.12],
        [0.39, 0.12, 0.25, 0.00, 0.13, 0.10],
        [0.36, 0.49, 0.09, 0.02, 0.00, 0.03],
        [0.15, 0.00, 0.80, 0.00, 0.05, 0.00]])
43_LL 90.27777777777779
0.9027777777777778
tensor([[0.00, 0.03, 0.42, 0.06, 0.33, 0.17],
        [0.00, 0.00, 0.00, 0.00, 0.00

### Calculate the consistency of weighted classifications

In [36]:


classification_types = ["Phyleticity", "Site clonality", "Genetic clonality", "Seeding pattern"]
for cidx in range(len(classification_types)):
    print(classification_types[cidx])
    # Calculate the consistency of each classification type
    pt_name_to_consistency_percentage = {}
    for pt_name in classifications_dict:
        all_pt_classications = classifications_dict[pt_name]
        classifications = [all_pt_classications[run][cidx] for run in range(NUM_RUNS)]
        #print(pt_name, classifications)
        mode_count = max([classifications.count(x) for x in classifications])
        mode_percentage = (mode_count / NUM_RUNS) * 100
        if mode_percentage != 100:
            print(pt_name, classifications)
        pt_name_to_consistency_percentage[run_name] = mode_percentage
        
    avg_consistency_percentage = sum(run_name_to_consistency_percentage.values()) / len(run_name_to_consistency_percentage)

    print("Average consistency percentage:")
    print(avg_consistency_percentage, "\n")

Phyleticity
56_LL ['monophyletic', 'monophyletic', 'polyphyletic']
31_LL ['monophyletic', 'monophyletic', 'polyphyletic']
17_LL ['polyphyletic', 'monophyletic', 'monophyletic']
61_LL ['polyphyletic', 'monophyletic', 'monophyletic']
26_LL ['monophyletic', 'polyphyletic', 'polyphyletic']
19_LL ['polyphyletic', 'polyphyletic', 'monophyletic']
10_LL ['monophyletic', 'polyphyletic', 'monophyletic']
Average consistency percentage:
95.29333333333334 

Site clonality
Average consistency percentage:
95.29333333333334 

Genetic clonality
Average consistency percentage:
95.29333333333334 

Seeding pattern
61_LL ['multi-source', 'reseeding', 'reseeding']
28_LL ['multi-source', 'reseeding', 'multi-source']
54_LL ['multi-source', 'multi-source', 'reseeding']
34_LL ['reseeding', 'multi-source', 'multi-source']
19_LL ['reseeding', 'multi-source', 'reseeding']
91_LL ['reseeding', 'multi-source', 'reseeding']
Average consistency percentage:
95.29333333333334 

